## Book Recommender System

## Step 1: Setup and Bronze Layer

In [1]:
!pip install -q pandas numpy pyarrow scipy scikit-learn

In [2]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np

from datetime import datetime, timezone
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors

base_dir = "/content/book_recommender"

bronze_dir = f"{base_dir}/bronze"
silver_dir = f"{base_dir}/silver"
gold_dir = f"{base_dir}/gold"

for folder in [bronze_dir, silver_dir, gold_dir]:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully")

Folders created successfully


In [5]:
import requests
import os
import zipfile

url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"

zip_path = f"{base_dir}/books.zip"
raw_dir = f"{base_dir}/raw"

os.makedirs(raw_dir, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0"
}

try:
    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(raw_dir)

    print("Dataset downloaded from the official source.")

except requests.exceptions.HTTPError as e:
    if response.status_code != 403:
        raise e

    print("Official source returned 403. Using GitHub mirror...")

    github_base = (
        "https://raw.githubusercontent.com/"
        "tttgm/fellowshipai/master/book_crossing_dataset/"
    )

    files = [
        "BX-Books.csv",
        "BX-Users.csv",
        "BX-Book-Ratings.csv"
    ]

    for filename in files:
        file_url = github_base + filename

        r = requests.get(file_url, headers=headers, timeout=120)
        r.raise_for_status()

        with open(f"{raw_dir}/{filename}", "wb") as f:
            f.write(r.content)

    print("Dataset downloaded successfully from GitHub.")

print("Files available:", os.listdir(raw_dir))

Dataset downloaded from the official source.
Files available: ['BX-Books.csv', 'BX-Book-Ratings.csv', 'BX-Users.csv']


In [8]:
import pandas as pd

users = pd.read_csv(
    find_file("BX-Users.csv"),
    sep=";",
    encoding="latin-1",
    low_memory=False
)

books = pd.read_csv(
    find_file("BX-Books.csv"),
    sep=";",
    encoding="latin-1",
    engine="python",
    on_bad_lines="warn"
)

ratings = pd.read_csv(
    find_file("BX-Book-Ratings.csv"),
    sep=";",
    encoding="latin-1",
    low_memory=False
)

print("Users shape:", users.shape)
print("Books shape:", books.shape)
print("Ratings shape:", ratings.shape)

display(books.head())

/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 815: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 1679: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 2531: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 2640: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 3236: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 3256: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 4906: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: ParserWarning: Skipping line 6452: ';' expected after '"'

  books = pd.read_csv(
/tmp/ipykernel_1053/1258697132.py:10: Par

Users shape: (278858, 3)
Books shape: (270491, 8)
Ratings shape: (1149780, 3)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [9]:
timestamp = datetime.now(timezone.utc).isoformat()

users_bronze = users.copy()
books_bronze = books.copy()
ratings_bronze = ratings.copy()

users_bronze["ingestion_timestamp"] = timestamp
books_bronze["ingestion_timestamp"] = timestamp
ratings_bronze["ingestion_timestamp"] = timestamp

users_bronze.to_parquet(f"{bronze_dir}/users.parquet", index=False)
books_bronze.to_parquet(f"{bronze_dir}/books.parquet", index=False)
ratings_bronze.to_parquet(f"{bronze_dir}/ratings.parquet", index=False)

print("Bronze layer completed")

Bronze layer completed


## Step 2: Exploratory Data Analysis (EDA)

In [10]:
users = pd.read_parquet(f"{bronze_dir}/users.parquet")
books = pd.read_parquet(f"{bronze_dir}/books.parquet")
ratings = pd.read_parquet(f"{bronze_dir}/ratings.parquet")

print("Users:", users.shape)
print("Books:", books.shape)
print("Ratings:", ratings.shape)

Users: (278858, 4)
Books: (270491, 9)
Ratings: (1149780, 4)


In [11]:
for name, df in [
    ("Users", users),
    ("Books", books),
    ("Ratings", ratings)
]:
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())


Users
Shape: (278858, 4)

Data Types:
User-ID                  int64
Location                object
Age                    float64
ingestion_timestamp     object
dtype: object

Missing Values:
User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64

Books
Shape: (270491, 9)

Data Types:
ISBN                   object
Book-Title             object
Book-Author            object
Year-Of-Publication     int64
Publisher              object
Image-URL-S            object
Image-URL-M            object
Image-URL-L            object
ingestion_timestamp    object
dtype: object

Missing Values:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            0
ingestion_timestamp    0
dtype: int64

Ratings
Shape: (1149780, 4)

Data Types:
User-ID                 int64
ISBN                   

In [12]:
print(ratings["Book-Rating"].value_counts().sort_index())

print("\nNumber of zero ratings:",
      (ratings["Book-Rating"] == 0).sum())

Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64

Number of zero ratings: 716109


In [13]:
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")

print(users["Age"].describe())

print("Age below 5:", (users["Age"] < 5).sum())
print("Age above 100:", (users["Age"] > 100).sum())
print("Missing ages:", users["Age"].isnull().sum())

count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64
Age below 5: 882
Age above 100: 366
Missing ages: 110762


In [14]:
books["Year-Of-Publication"] = pd.to_numeric(
    books["Year-Of-Publication"],
    errors="coerce"
)

print(books["Year-Of-Publication"].describe())

print("Year 0:",
      (books["Year-Of-Publication"] == 0).sum())

print("Future years:",
      (books["Year-Of-Publication"] > datetime.now().year).sum())

count    270491.000000
mean       1959.704914
std         258.207234
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Name: Year-Of-Publication, dtype: float64
Year 0: 4611
Future years: 11


In [15]:
user_ratings = ratings.groupby("User-ID").size()
book_ratings = ratings.groupby("ISBN").size()

print("Ratings per user:")
print(user_ratings.describe())

print("\nRatings per book:")
print(book_ratings.describe())

Ratings per user:
count    105283.000000
mean         10.920851
std          90.562825
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max       13602.000000
dtype: float64

Ratings per book:
count    340556.000000
mean          3.376185
std          12.436252
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        2502.000000
dtype: float64


In [16]:
total_users = ratings["User-ID"].nunique()
total_books = ratings["ISBN"].nunique()

total_possible = total_users * total_books

actual_ratings = ratings[
    ["User-ID", "ISBN"]
].drop_duplicates().shape[0]

sparsity = (1 - actual_ratings / total_possible) * 100

print("Total possible ratings:", total_possible)
print("Actual ratings:", actual_ratings)
print("Matrix sparsity:", round(sparsity, 2), "%")

Total possible ratings: 35854757348
Actual ratings: 1149780
Matrix sparsity: 100.0 %



## EDA Observations

1. **Users:** The dataset contains 278,858 users. The Age column has 110,762 missing values, approximately 39.7% of the records. There are also 882 ages below 5 and 366 ages above 100. In the Silver layer, invalid ages will be replaced with null values and missing ages will be filled using the median.

2. **Books:** The dataset contains 270,491 books. Book-Author and Publisher have 2 missing values each. Publication years contain 4,611 zero values and 11 future years. These invalid publication years will be replaced with null values, and missing text values will be handled during cleaning.

3. **Ratings:** There are 1,149,780 ratings. Out of these, 716,109 ratings are zero, representing approximately 62.3% of all ratings. Since zero represents an implicit rating rather than an explicit score, these records will be removed. Only ratings between 1 and 10 will be retained.

4. **Ratings per User:** The average number of ratings per user is approximately 10.92, while the median is only 1. This indicates that most users have rated very few books, while a small number of users are highly active.

5. **Ratings per Book:** The average number of ratings per book is approximately 3.38, and the median is 1. This indicates that most books have very few ratings, resulting in sparse user-book interactions.

6. **Matrix Sparsity:** The user-book matrix has 35,854,757,348 possible interactions but only 1,149,780 actual ratings. Its sparsity is approximately 99.997% (displayed as 100.0% due to rounding). This makes a sparse matrix representation suitable for the recommendation system.

### Conclusion

The dataset contains missing values, invalid records, zero ratings, and highly sparse user-book interactions. In the Silver layer, we will clean these issues, remove invalid ratings, handle missing values, and filter users and books with fewer than 10 ratings to improve the quality of recommendations.


## Step 3: Data Cleaning and Silver Layer

In [17]:
users = users.drop_duplicates()
books = books.drop_duplicates()
ratings = ratings.drop_duplicates()

print("Duplicates removed")
print(users.shape, books.shape, ratings.shape)

Duplicates removed
(278858, 4) (270491, 9) (1149780, 4)


In [18]:
ratings["Book-Rating"] = pd.to_numeric(
    ratings["Book-Rating"],
    errors="coerce"
)

ratings = ratings[
    ratings["Book-Rating"].between(1, 10)
].copy()

print("Ratings after cleaning:", ratings.shape)
print(ratings["Book-Rating"].value_counts().sort_index())

Ratings after cleaning: (433671, 4)
Book-Rating
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64


In [19]:
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")

users.loc[
    (users["Age"] < 5) | (users["Age"] > 100),
    "Age"
] = np.nan

median_age = users["Age"].median()

users["Age"] = users["Age"].fillna(median_age)

print("Missing ages:", users["Age"].isnull().sum())
print("Median age:", median_age)

Missing ages: 0
Median age: 32.0


In [20]:
location = users["Location"].fillna("").astype(str).str.split(
    ",", n=2, expand=True
)

users["City"] = location[0].str.strip()

users["State"] = (
    location[1].str.strip()
    if 1 in location.columns else np.nan
)

users["Country"] = (
    location[2].str.strip()
    if 2 in location.columns else np.nan
)

print(users[["Location", "City", "State", "Country"]].head())

                             Location         City            State  \
0                  nyc, new york, usa          nyc         new york   
1           stockton, california, usa     stockton       california   
2     moscow, yukon territory, russia       moscow  yukon territory   
3           porto, v.n.gaia, portugal        porto         v.n.gaia   
4  farnborough, hants, united kingdom  farnborough            hants   

          Country  
0             usa  
1             usa  
2          russia  
3        portugal  
4  united kingdom  


In [21]:
books["Year-Of-Publication"] = pd.to_numeric(
    books["Year-Of-Publication"],
    errors="coerce"
)

current_year = datetime.now().year

books.loc[
    (books["Year-Of-Publication"] <= 0) |
    (books["Year-Of-Publication"] > current_year),
    "Year-Of-Publication"
] = np.nan

print("Invalid years remaining:",
      ((books["Year-Of-Publication"] <= 0) |
       (books["Year-Of-Publication"] > current_year)).sum())

Invalid years remaining: 0


In [22]:
text_columns = [
    "Book-Title",
    "Book-Author",
    "Publisher"
]

for col in text_columns:
    books[col] = books[col].astype("string").str.strip()

image_columns = [
    col for col in books.columns
    if col.lower().startswith("image-url")
]

books.drop(columns=image_columns, inplace=True)

print("Text cleaned")
print("Remaining columns:", books.columns.tolist())

Text cleaned
Remaining columns: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'ingestion_timestamp']


In [23]:
users["User-ID"] = pd.to_numeric(
    users["User-ID"], errors="coerce"
)

ratings["User-ID"] = pd.to_numeric(
    ratings["User-ID"], errors="coerce"
)

users = users.dropna(subset=["User-ID"])
ratings = ratings.dropna(subset=["User-ID", "ISBN", "Book-Rating"])
books = books.dropna(subset=["ISBN"])

users["User-ID"] = users["User-ID"].astype(int)
ratings["User-ID"] = ratings["User-ID"].astype(int)

users = users.drop_duplicates(subset=["User-ID"])
books = books.drop_duplicates(subset=["ISBN"])

ratings = ratings[
    ratings["User-ID"].isin(users["User-ID"]) &
    ratings["ISBN"].isin(books["ISBN"])
].copy()

print("Valid users:", len(users))
print("Valid books:", len(books))
print("Valid ratings:", len(ratings))

Valid users: 278858
Valid books: 270491
Valid ratings: 382827


### Minimum Rating Threshold

In [24]:
min_ratings = 10

while True:
    user_counts = ratings.groupby("User-ID").size()
    book_counts = ratings.groupby("ISBN").size()

    valid_users = user_counts[user_counts >= min_ratings].index
    valid_books = book_counts[book_counts >= min_ratings].index

    old_shape = ratings.shape

    ratings = ratings[
        ratings["User-ID"].isin(valid_users) &
        ratings["ISBN"].isin(valid_books)
    ]

    users = users[users["User-ID"].isin(valid_users)]
    books = books[books["ISBN"].isin(valid_books)]

    if ratings.shape == old_shape:
        break

print("Users:", len(users))
print("Books:", len(books))
print("Ratings:", len(ratings))

print("Minimum user ratings:",
      ratings.groupby("User-ID").size().min())

print("Minimum book ratings:",
      ratings.groupby("ISBN").size().min())

Users: 1813
Books: 2022
Ratings: 41238
Minimum user ratings: 10
Minimum book ratings: 10


### Saving Silver Data

In [25]:
users.to_parquet(f"{silver_dir}/users.parquet", index=False)
books.to_parquet(f"{silver_dir}/books.parquet", index=False)
ratings.to_parquet(f"{silver_dir}/ratings.parquet", index=False)

print("Silver layer completed")

Silver layer completed


## Step 4: Gold Layer

In [26]:
user_ids = sorted(ratings["User-ID"].unique())
book_ids = sorted(ratings["ISBN"].unique())

user_to_idx = {
    user_id: idx for idx, user_id in enumerate(user_ids)
}

book_to_idx = {
    book_id: idx for idx, book_id in enumerate(book_ids)
}

print("Number of users:", len(user_ids))
print("Number of books:", len(book_ids))

Number of users: 1813
Number of books: 2022


In [27]:
row = ratings["ISBN"].map(book_to_idx)
col = ratings["User-ID"].map(user_to_idx)
data = ratings["Book-Rating"]

rating_matrix = csr_matrix(
    (data, (row, col)),
    shape=(len(book_ids), len(user_ids))
)

print("Matrix shape:", rating_matrix.shape)
print("Non-zero values:", rating_matrix.nnz)

Matrix shape: (2022, 1813)
Non-zero values: 41238


In [28]:
save_npz(
    f"{gold_dir}/book_user_rating_matrix.npz",
    rating_matrix
)

pd.DataFrame({
    "ISBN": book_ids,
    "book_idx": range(len(book_ids))
}).to_csv(f"{gold_dir}/book_index.csv", index=False)

pd.DataFrame({
    "User-ID": user_ids,
    "user_idx": range(len(user_ids))
}).to_csv(f"{gold_dir}/user_index.csv", index=False)

print("Rating matrix and mappings saved")

Rating matrix and mappings saved


### Book Lookup Table

In [29]:
book_lookup = books[
    ["ISBN", "Book-Title", "Book-Author"]
].drop_duplicates(subset=["ISBN"])

book_lookup.to_csv(
    f"{gold_dir}/book_lookup.csv",
    index=False
)

display(book_lookup.head())

,ISBN,Book-Title,Book-Author
18,0440234743,The Testament,John Grisham
19,0452264464,Beloved (Plume Contemporary Fiction),Toni Morrison
26,0971880107,Wild Animus,Rich Shapero
27,0345402871,Airframe,Michael Crichton
28,0345417623,Timeline,MICHAEL CRICHTON


### User-Level and Book-Level Aggregations


In [30]:
user_aggregates = ratings.groupby("User-ID")["Book-Rating"].agg(
    average_rating="mean",
    number_of_ratings="count"
).reset_index()

book_aggregates = ratings.groupby("ISBN")["Book-Rating"].agg(
    average_rating="mean",
    number_of_ratings="count"
).reset_index()

user_aggregates.to_csv(
    f"{gold_dir}/user_aggregates.csv",
    index=False
)

book_aggregates.to_csv(
    f"{gold_dir}/book_aggregates.csv",
    index=False
)

display(user_aggregates.head())
display(book_aggregates.head())

print("Gold layer completed")

,User-ID,average_rating,number_of_ratings
0,243,7.769231,13
1,254,8.450000,20
2,638,9.454545,22
3,805,8.400000,10
4,882,8.235294,17


,ISBN,average_rating,number_of_ratings
0,0020199600,8.090909,11
1,0020442203,8.833333,12
2,002542730X,7.481481,27
3,0028604199,7.900000,10
4,0060002050,7.545455,11


Gold layer completed


## Step 5: Building the Recommender System

In [31]:
rating_matrix = load_npz(
    f"{gold_dir}/book_user_rating_matrix.npz"
)

book_index = pd.read_csv(f"{gold_dir}/book_index.csv")
user_index = pd.read_csv(f"{gold_dir}/user_index.csv")

book_lookup = pd.read_csv(
    f"{gold_dir}/book_lookup.csv",
    dtype={"ISBN": str}
)

user_to_idx = dict(zip(
    user_index["User-ID"],
    user_index["user_idx"]
))

book_to_idx = dict(zip(
    book_index["ISBN"],
    book_index["book_idx"]
))

idx_to_book = dict(zip(
    book_index["book_idx"],
    book_index["ISBN"]
))

book_info = book_lookup.set_index("ISBN").to_dict("index")

print("Gold data loaded successfully")

Gold data loaded successfully


### Cosine Similarity Model


In [32]:
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

model.fit(rating_matrix)

print("Cosine similarity model trained")

Cosine similarity model trained


### Finding Similar Books

In [33]:
def similar_books(book_title, n=10):

    matches = book_lookup[
        book_lookup["Book-Title"].str.lower() ==
        book_title.lower()
    ]

    if matches.empty:
        return pd.DataFrame()

    isbn = matches.iloc[0]["ISBN"]

    if isbn not in book_to_idx:
        return pd.DataFrame()

    book_idx = book_to_idx[isbn]

    distances, indices = model.kneighbors(
        rating_matrix[book_idx],
        n_neighbors=min(n + 1, rating_matrix.shape[0])
    )

    recommendations = []

    for distance, idx in zip(distances[0], indices[0]):

        if idx == book_idx:
            continue

        isbn = idx_to_book[idx]
        info = book_info.get(isbn, {})

        recommendations.append({
            "Book-Title": info.get("Book-Title"),
            "Book-Author": info.get("Book-Author"),
            "Similarity": round(1 - distance, 4)
        })

    return pd.DataFrame(recommendations).head(n)

In [34]:
book_title = book_lookup.iloc[0]["Book-Title"]

print("Selected book:", book_title)

display(similar_books(book_title, n=10))

Selected book: The Testament


,Book-Title,Book-Author,Similarity
0,Monster : A Novel (Alex Delaware Novels (Paper...,JONATHAN KELLERMAN,0.2528
1,The Runaway Jury,JOHN GRISHAM,0.2378
2,The Bone Collector (Lincoln Rhyme Novels (Pape...,Jeff Deaver,0.2376
3,Call of the Wild,Jack London,0.2359
4,The Simple Truth,David Baldacci,0.2356
5,The Street Lawyer,JOHN GRISHAM,0.2198
6,Atlantis Found,Clive Cussler,0.1994
7,Outbreak,Robin Cook,0.1987
8,From Potter's Field,Patricia Daniels Cornwell,0.1962
9,Writ of Execution,Perri O'Shaughnessy,0.1901


### Generating User Recommendations


In [35]:
def recommend(user_id, n=10):

    if user_id not in user_to_idx:
        return pd.DataFrame()

    user_idx = user_to_idx[user_id]

    user_ratings = rating_matrix[:, user_idx].tocoo()

    already_rated = set(user_ratings.row)

    liked_books = [
        (idx, rating)
        for idx, rating in zip(
            user_ratings.row,
            user_ratings.data
        )
        if rating >= 7
    ]

    scores = {}

    for book_idx, rating in liked_books:

        distances, indices = model.kneighbors(
            rating_matrix[book_idx],
            n_neighbors=min(31, rating_matrix.shape[0])
        )

        for distance, similar_idx in zip(
            distances[0], indices[0]
        ):

            if similar_idx in already_rated:
                continue

            similarity = 1 - distance

            scores[similar_idx] = (
                scores.get(similar_idx, 0) +
                similarity * rating
            )

    ranked_books = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:n]

    results = []

    for idx, score in ranked_books:

        isbn = idx_to_book[idx]
        info = book_info.get(isbn, {})

        results.append({
            "Book-Title": info.get("Book-Title"),
            "Book-Author": info.get("Book-Author"),
            "Score": round(score, 4)
        })

    return pd.DataFrame(results)

## Step 6: Demo and Sanity Check


In [36]:
user_counts = ratings.groupby("User-ID").size()

demo_users = []

for user_id in user_counts.sort_values(ascending=False).index:

    user_idx = user_to_idx[user_id]

    user_ratings = rating_matrix[:, user_idx].tocoo()

    if any(r >= 7 for r in user_ratings.data):
        demo_users.append(user_id)

    if len(demo_users) == 3:
        break

print("Selected users:", demo_users)

Selected users: [11676, 16795, 95359]


In [38]:
for user_id in demo_users:

    print("\n" + "=" * 60)
    print("USER ID:", user_id)

    user_idx = user_to_idx[user_id]

    user_ratings = rating_matrix[:, user_idx].tocoo()

    liked_books = []

    for idx, rating in zip(
        user_ratings.row,
        user_ratings.data
    ):

        if rating >= 7:

            isbn = idx_to_book[idx]
            info = book_info.get(isbn, {})

            liked_books.append({
                "Book-Title": info.get("Book-Title"),
                "Rating": rating
            })

    print("\nHighly Rated Books:")
    display(
        pd.DataFrame(liked_books)
        .sort_values("Rating", ascending=False)
        .head(10)
    )

    print("\nRecommended Books:")
    display(recommend(user_id, n=10))


USER ID: 11676

Highly Rated Books:


,Book-Title,Rating
9,I Know This Much Is True (Oprah's Book Club),10
731,The Time Traveler's Wife (Today Show Book Club...,10
730,The Time Traveler's Wife,10
11,The Poisonwood Bible,10
726,Tipping the Velvet,10
21,To Kill a Mockingbird,10
4,The Poisonwood Bible,10
5,Daughter of Fortune : A Novel (Oprah's Book Cl...,10
340,Mixed Blessings,10
314,Harry Potter and the Chamber of Secrets (Book 2),10



Recommended Books:


,Book-Title,Book-Author,Score
0,"Night Shield (Intimate Moments, 1027)",Nora Roberts,60.3088
1,Public Secrets,Nora Roberts,48.6376
2,Palomino,DANIELLE STEEL,46.2867
3,Night Tales,Nora Roberts,44.6990
4,Birthright,Nora Roberts,44.0750
5,A Perfect Stranger,Danielle Steel,42.7191
6,Tears of the Moon (Irish Trilogy),Nora Roberts,41.2279
7,Stanislaski Brothers (Silhouette Promo),Nora Roberts,38.7562
8,Conspiracy in Death,J. D. Robb,36.5470
9,Born in Ice,Nora Roberts,36.5326



USER ID: 16795

Highly Rated Books:


,Book-Title,Rating
23,The Lovely Bones: A Novel,10
20,The Dogs of Babel (Today Show Book Club #12),10
14,The Samurai's Garden : A Novel,10
12,The Secret Life of Bees,10
13,Life of Pi,10
9,The Liar's Club: A Memoir,10
60,The Da Vinci Code,10
61,Skipping Christmas,10
62,Skipping Christmas,10
41,The Dive From Clausen's Pier (Alex Awards),10


... (2 outputs left out)


USER ID: 95359

Highly Rated Books:


,Book-Title,Rating
2,Stupid White Men ...and Other Sorry Excuses fo...,10
28,The Hours : A Novel,10
23,"Word Freak: Heartbreak, Triumph, Genius, and O...",10
22,Year of Wonders,10
30,The Hours: A Novel,10
18,The Westing Game (Now in Speak!),10
84,A Wrinkle In Time,10
100,The Handmaid's Tale,10
141,Midnight in the Garden of Good and Evil,10
153,Thousand Acres,10



Recommended Books:


,Book-Title,Book-Author,Score
0,Tipping the Velvet,Sarah Waters,21.2261
1,Season of the Machete,James Patterson,18.9590
2,On the Banks of Plum Creek,Laura Ingalls Wilder,16.4869
3,The Phantom Tollbooth,Norton Juster,16.3653
4,Hornet's Nest,Patricia Daniels Cornwell,16.1094
5,Small Sacrifices: A True Story of Passion and ...,Ann Rule,15.9259
6,Waiting (Vintage International),Ha Jin,14.3584
7,The Shell Seekers (Shell Seekers),Rosamunde Pilcher,14.2315
8,The Professor and the Madman,Simon Winchester,13.3271
9,Sea Glass: A Novel,Anita Shreve,13.3235


In [39]:
for user_id in demo_users:

    user_idx = user_to_idx[user_id]

    rated_indices = set(
        rating_matrix[:, user_idx].tocoo().row
    )

    recommendations = recommend(user_id, n=10)

    print(
        f"User {user_id}: "
        f"{len(recommendations)} recommendations generated"
    )

    print(
        "Recommendations generated successfully:",
        not recommendations.empty
    )

User 11676: 10 recommendations generated
Recommendations generated successfully: True
User 16795: 10 recommendations generated
Recommendations generated successfully: True
User 95359: 10 recommendations generated
Recommendations generated successfully: True



## Recommendation Results and Observations

The recommendation system was tested on three selected users using their highly rated books as input. For each user, the system generated 10 book recommendations based on item-item cosine similarity and weighted similarity scores.

### User 11676
- The user has highly rated books such as *The Time Traveler's Wife*, *The Poisonwood Bible*, and *To Kill a Mockingbird*.
- The system recommended several books by Nora Roberts and Danielle Steel, including *Night Shield*, *Public Secrets*, and *Palomino*.
- This indicates that the system identified similarities between the user's reading preferences and books by these authors.

### User 16795
- The user has highly rated books such as *The Lovely Bones*, *The Secret Life of Bees*, *Life of Pi*, and *The Da Vinci Code*.
- Recommendations included *Message from Nam*, *Thurston House*, *The Endearment*, and *Midnight Bayou*.
- The recommendations include books from authors such as Danielle Steel, Nora Roberts, and LaVyrle Spencer.

### User 95359
- The user has highly rated books such as *The Hours*, *Year of Wonders*, *The Handmaid's Tale*, and *A Wrinkle in Time*.
- Recommendations included *Tipping the Velvet*, *Season of the Machete*, *The Phantom Tollbooth*, and *The Professor and the Madman*.
- The system generated recommendations from a diverse set of authors and titles based on the user's previously rated books.

### Overall Observations

1. The recommendation system successfully generated 10 recommendations for each of the three selected users.
2. Recommendations were generated using item-based collaborative filtering and cosine similarity between books.
3. Highly rated books (rating >= 7) were used to identify the user's reading preferences.
4. Similarity scores were weighted by the user's ratings to rank candidate books.
5. Books already rated by a user were excluded from their recommendations.
6. Different users received different recommendations based on their individual rating histories.

### Conclusion

The results demonstrate that the item-based collaborative filtering system can generate personalized book recommendations using historical user ratings. The Medallion Architecture helped organize the data into Bronze, Silver, and Gold layers, while the Gold layer provided the cleaned data and sparse user-book matrix required for recommendation generation.


## Step 7: Saving the Final Outputs

In [40]:
for folder in [bronze_dir, silver_dir, gold_dir]:

    print("\n", folder)

    for file in os.listdir(folder):
        print(file)


 /content/book_recommender/bronze
books.parquet
users.parquet
ratings.parquet

 /content/book_recommender/silver
books.parquet
users.parquet
ratings.parquet

 /content/book_recommender/gold
book_index.csv
book_lookup.csv
book_aggregates.csv
user_index.csv
book_user_rating_matrix.npz
user_aggregates.csv


In [41]:
import shutil

submission_dir = "/content/submission"

os.makedirs(submission_dir, exist_ok=True)

for layer in ["bronze", "silver", "gold"]:

    shutil.copytree(
        f"{base_dir}/{layer}",
        f"{submission_dir}/{layer}",
        dirs_exist_ok=True
    )

shutil.make_archive(
    "/content/book_recommender_submission",
    "zip",
    submission_dir
)

print("Submission ZIP created")

Submission ZIP created


In [42]:
from google.colab import files

files.download("/content/book_recommender_submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>